# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()


# 18. 写真から部屋の形を作る —— 調査の報告(2026-10-06。**実装しない**)

利用者の質問「ストリートビューの画像で、更に正確な部屋構造を作成することは可能か」への調査。
**結論: 部屋ごとの大まかな形(四角い部屋の壁の位置と向き)なら、360° の写真 1 枚からでも作れる見込みがある。
建物全体の正確な図面は、写真だけでは難しい。** 以下は公開の研究・道具の一般的な性質に基づく整理で、**この学校の写真で試してはいない**。

## 1. やり方の候補

| やり方 | 要るもの | 得られるもの | 向き・不向き |
|---|---|---|---|
| **360° 写真からの間取りの推定**(HorizonNet・LED²-Net・HoHoNet などの研究) | 部屋の中から撮った 360° 写真 1 枚 | 部屋の角の位置(床と天井の境の線)。**壁が直角に交わる部屋**なら形になる | ストリートビューの写真をそのまま使える。**大きさは「カメラの高さ」を知らないと決まらない**(撮るときに三脚の高さを決めておく) |
| **写真測量**(COLMAP・Meshroom・RealityCapture など) | 1 部屋あたり数十〜数百枚の重なった写真 | 3 次元の点群・面 | 細かい形まで出るが、**撮影の手間が大きい**。白い壁・ガラスは苦手。処理は GPU のある PC(VPS では無理) |
| **LiDAR 付きの端末で測る**(iPhone Pro の RoomPlan など) | LiDAR 付きの iPhone / iPad | 部屋の寸法・壁・扉・窓が直接出る | **いちばん正確で速い**が、端末が要る。Android の深度(ARCore Depth)は精度が落ちる |
| 今の見取り図(PNG)をなぞる | 図面 | 壁の線 | 図面が正しければこれが一番安い。写真は「図面と現物が違う所」を見つけるのに使う |

## 2. この地図にどう生かせるか

- **北と距離の補正(`MapCalibration`)の手がかり**: 写真の向き(`heading`)と推定した壁の向きから、その部屋の壁が北から何度かが出る。
  見取り図の壁の向きと比べれば、階ごとの「地図の上が北から何度か」を確かめられる
- **縮尺(px/m)の手がかり**: カメラの高さを決めて撮れば、部屋の幅が m で出る。見取り図の画素と比べて px/m を確かめられる
- **見取り図と現物の食い違いの発見**: 推定した部屋の形を見取り図に重ね、大きく違う所(改修・仕切りの追加)を拾う
- 地点の位置(ノードの x, y)を写真から決め直すのは、写真の撮影位置が正確に分からないと難しい

## 3. 進めるなら(案)

1. 形の単純な部屋を 3〜5 つ選び、**三脚で高さを決めて** 360° 写真を撮る(今の `api/panorama.php` にそのまま上げられる)
2. PC(GPU があると速い)で間取りの推定を流し、部屋の幅・奥行き・壁の向きを出す
3. 見取り図の同じ部屋と比べ、**ずれが何 cm・何度か**を表にする —— これで「使えるか」を決める
4. 使えるなら、管理画面かアプリに「推定した部屋の形を見取り図に重ねて見る」を足す(自動で地図を書き換えはしない)

## 4. 決めておくこと

- 撮影の決まり(三脚の高さ・最初の向きを北にそろえるか)
- 推定を流す PC(VPS では重すぎる。メモリ 2GB では動かない)
- 研究の道具の使用条件(多くは研究用。使う前にライセンスを確かめる)